In [22]:
from ultralytics import solutions
import cv2
import numpy as np

points = []

In [23]:
# cam= cv2.VideoCapture(0)

In [24]:
stream_url = "https://strm1.spatic.go.kr/live/36.stream/chunklist_w626582203.m3u8"
cam = cv2.VideoCapture(stream_url)

In [25]:
def mouse_callback(event, x, y, flags, param):
    """
    마우스 이벤트 콜백 함수
    - 좌 클릭 시 좌표를 points 리스트에 추가
    - 우 클릭 시 마지막 좌표를 삭제
    - 좌표 출력
    """
    if event == cv2.EVENT_LBUTTONDOWN: # 좌 클릭 이벤트 감지
        points.append((x, y))
        print(f"클릭된 좌표는 {x, y} 입니다.")
    elif event == cv2.EVENT_RBUTTONDOWN and points: # 우 클릭 이벤트 감지
        removed = points.pop()
        print(f"{removed} 좌표를 삭제했습니다.")

In [26]:
# WINDOW_AUTOSIZE: 창 크기 = 영상 크기(640x480) 고정
# 창을 늘리면 클릭 좌표와 영상 좌표가 어긋나므로 크기 조절 금지
cv2.namedWindow("GET_X_Y", cv2.WINDOW_AUTOSIZE)
cv2.setMouseCallback("GET_X_Y", mouse_callback)

In [27]:
# 영상을 보면서 감지 영역의 꼭짓점을 클릭으로 찍는다
# 좌 클릭: 점 추가 / 우 클릭: 마지막 점 삭제 / r: 전체 초기화 / s: 확정 후 종료
while cam.isOpened():
    success, frame = cam.read()
    if not success:
        print("비디오 읽기 실패 . . .")
        break

    # 아래 감지 루프와 같은 크기로 맞춰야 좌표가 일치한다
    re_frame = cv2.resize(frame, (640, 480))

    # 지금까지 찍은 점과 다각형을 미리보기로 그린다
    for point in points:
        cv2.circle(re_frame, point, 5, (0, 0, 255), -1)
    if len(points) >= 2:
        cv2.polylines(re_frame, [np.array(points)], isClosed=len(points) >= 3,
                      color=(0, 255, 0), thickness=2)

    cv2.imshow("GET_X_Y", re_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        if len(points) < 3:
            print(f"다각형은 최소 3점이 필요합니다. 현재 {len(points)}점")
            continue
        print(f"영역 확정: {points}")
        break
    elif key == ord('r'):
        points.clear()
        print("좌표를 모두 초기화했습니다.")
    elif key == ord('q'):
        print("q키를 눌러서 종료")
        break

cv2.destroyWindow("GET_X_Y")

클릭된 좌표는 (199, 138) 입니다.
클릭된 좌표는 (141, 257) 입니다.
클릭된 좌표는 (217, 318) 입니다.
클릭된 좌표는 (371, 331) 입니다.
클릭된 좌표는 (454, 153) 입니다.
클릭된 좌표는 (372, 62) 입니다.
q키를 눌러서 종료


In [28]:
# 클릭으로 얻은 좌표를 RegionCounter가 받는 형식으로 변환
region_points = {
    "region-01": points
}
print(region_points)

{'region-01': [(199, 138), (141, 257), (217, 318), (371, 331), (454, 153), (372, 62)]}


In [29]:
yolo_region = solutions.RegionCounter(
    model="yolo11n.pt",
    show=False,
    region=region_points,
    conf=0.1          
)

Ultralytics Solutions:  {'source': None, 'model': 'yolo11n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': {'region-01': [(199, 138), (141, 257), (217, 318), (371, 331), (454, 153), (372, 62)]}, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.1, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


In [30]:
while cam.isOpened():
    success, frame = cam.read()
    if not success:
        print("비디오 읽기 실패 . . .")
        break

 
    re_frame = cv2.resize(frame, (640, 480))

    results = yolo_region(re_frame)

    cv2.imshow("REGION", results.plot_im)

    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("q키를 눌러서 종료")
        break


0: 640x640 3.3ms, 9 car, 1 truck, 1 train, 1 bus
Speed: 139.2ms track, 3.3ms solution per image at shape (1, 3, 640, 640)

1: 640x640 3.0ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 104.4ms track, 3.0ms solution per image at shape (1, 3, 640, 640)

2: 640x640 2.9ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 110.6ms track, 2.9ms solution per image at shape (1, 3, 640, 640)

3: 640x640 2.9ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 107.1ms track, 2.9ms solution per image at shape (1, 3, 640, 640)

4: 640x640 3.1ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 118.5ms track, 3.1ms solution per image at shape (1, 3, 640, 640)

5: 640x640 3.0ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 122.1ms track, 3.0ms solution per image at shape (1, 3, 640, 640)

6: 640x640 3.4ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 128.1ms track, 3.4ms solution per image at shape (1, 3, 640, 640)

7: 640x640 3.7ms, 8 car, 1 truck, 1 train, 1 bus
Speed: 129.7ms track, 3.7ms solution per image at shape (1, 3, 640, 640)

8: 640x640 3.4ms

In [31]:
cam.release()
cv2.destroyAllWindows()